# Training Health Monitor - Colab Demo 🏥

**A comprehensive framework for monitoring and optimizing neural network training**

---

## 📋 Setup Instructions

This notebook demonstrates the complete Training Health Monitor system with:
- Real-time issue detection
- Intelligent recommendations
- Comprehensive reporting
- Support for PyTorch and TensorFlow


## Step 1: Clone and Install

In [ ]:
# Clone the repository
!git clone https://github.com/mohmed1983mohme-cyber/training-health-monitor.git
!cd training-health-monitor && pip install -q -r requirements.txt

## Step 2: Import Libraries

In [ ]:
import sys
sys.path.insert(0, '/content/training-health-monitor')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Import our framework
from training_monitor import TrainingHealthMonitor
from training_monitor.utils import extract_gradients_pytorch
from training_monitor.visualizer import Visualizer
from training_monitor.reporter import Reporter

print("✅ All imports successful!")

## Step 3: Create a Simple Neural Network

In [ ]:
class SimpleNet(nn.Module):
    """Simple neural network for demonstration"""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNet().to(device)
print(f"✅ Model created on device: {device}")
print(f"\nModel Architecture:")
print(model)

## Step 4: Create Synthetic Dataset

In [ ]:
# Create synthetic data
print("Creating synthetic dataset...")
X_train = torch.randn(1000, 1, 28, 28)
y_train = torch.randint(0, 10, (1000,))
X_val = torch.randn(200, 1, 28, 28)
y_val = torch.randint(0, 10, (200,))

# Create data loaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print(f"✅ Training samples: {len(train_dataset)}")
print(f"✅ Validation samples: {len(val_dataset)}")
print(f"✅ Batch size: 32")

## Step 5: Initialize Training Health Monitor

In [ ]:
# Initialize optimizer and loss function
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Initialize monitor
monitor = TrainingHealthMonitor(
    model=model,
    framework='pytorch',
    verbose=True,
    thresholds={
        'overfitting_ratio': 1.2,
        'underfitting_threshold': 0.5,
        'gradient_threshold': 10.0,
    }
)

print("✅ Training Health Monitor initialized!")
print(f"   Framework: PyTorch")
print(f"   Device: {device}")

## Step 6: Training Loop with Health Monitoring

In [ ]:
num_epochs = 10
training_history = []

print("\n" + "="*80)
print("🚀 STARTING TRAINING WITH HEALTH MONITORING")
print("="*80 + "\n")

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0
    train_acc = 0
    num_batches = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predictions = torch.max(outputs, 1)
        train_acc += (predictions == y_batch).sum().item() / len(y_batch)
        num_batches += 1
    
    train_loss /= num_batches
    train_acc /= num_batches
    
    # Validation phase
    model.eval()
    val_loss = 0
    val_acc = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()
            
            _, predictions = torch.max(outputs, 1)
            val_acc += (predictions == y_batch).sum().item() / len(y_batch)
    
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)
    
    # Extract gradients
    gradients = extract_gradients_pytorch(model)
    
    # Check health
    health = monitor.check_health(
        train_loss=train_loss,
        val_loss=val_loss,
        train_metrics={'accuracy': train_acc},
        val_metrics={'accuracy': val_acc},
        epoch=epoch,
        gradients=gradients
    )
    
    training_history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'status': health.status
    })
    
    # Print summary
    status_emoji = '✅' if health.status == 'healthy' else '⚠️' if health.status == 'warning' else '🔴'
    print(f"Epoch {epoch+1:2d}/{num_epochs} {status_emoji}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"  Health: {health.status.upper()}")
    if health.issues:
        print(f"  Issues: {', '.join(health.issues)}")
    print()

print("="*80)
print("✅ TRAINING COMPLETED")
print("="*80)

## Step 7: Generate Training Summary

In [ ]:
# Get summary
summary = monitor.get_summary()

print("\n" + "="*80)
print("📊 TRAINING HEALTH SUMMARY")
print("="*80)
print(f"\n Total Epochs:        {summary['total_epochs']}")
print(f" ✅ Healthy Epochs:   {summary['healthy']} ({summary['healthy']/summary['total_epochs']*100:.1f}%)")
print(f" ⚠️  Warning Epochs:   {summary['warnings']} ({summary['warnings']/summary['total_epochs']*100:.1f}%)")
print(f" 🔴 Critical Epochs:  {summary['critical']} ({summary['critical']/summary['total_epochs']*100:.1f}%)")
print(f"\n 📈 Overall Health Score: {summary['health_percentage']:.1f}%")
print("\n" + "="*80)

## Step 8: Visualize Training Progress

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Training Health Monitor - Results', fontsize=16, fontweight='bold')

# 1. Loss Curves
ax = axes[0, 0]
epochs = [h['epoch'] for h in training_history]
train_losses = [h['train_loss'] for h in training_history]
val_losses = [h['val_loss'] for h in training_history]
ax.plot(epochs, train_losses, 'o-', label='Train Loss', linewidth=2, markersize=8)
ax.plot(epochs, val_losses, 's-', label='Val Loss', linewidth=2, markersize=8)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Curves', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# 2. Accuracy Curves
ax = axes[0, 1]
train_accs = [h['train_acc'] for h in training_history]
val_accs = [h['val_acc'] for h in training_history]
ax.plot(epochs, train_accs, 'o-', label='Train Acc', linewidth=2, markersize=8)
ax.plot(epochs, val_accs, 's-', label='Val Acc', linewidth=2, markersize=8)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy Curves', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# 3. Loss Ratio (Overfitting Indicator)
ax = axes[1, 0]
loss_ratios = [h['val_loss'] / h['train_loss'] if h['train_loss'] > 0 else 0 for h in training_history]
colors = ['green' if r < 1.2 else 'orange' if r < 1.5 else 'red' for r in loss_ratios]
ax.bar(epochs, loss_ratios, color=colors, alpha=0.7, edgecolor='black')
ax.axhline(y=1.2, color='r', linestyle='--', linewidth=2, label='Overfitting Threshold')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Val Loss / Train Loss', fontsize=12)
ax.set_title('Overfitting Indicator', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# 4. Health Status
ax = axes[1, 1]
statuses = [h['status'] for h in training_history]
status_counts = {'healthy': statuses.count('healthy'), 
                 'warning': statuses.count('warning'),
                 'critical': statuses.count('critical')}
colors_pie = ['green', 'orange', 'red']
ax.pie([status_counts['healthy'], status_counts['warning'], status_counts['critical']], 
       labels=['Healthy', 'Warning', 'Critical'],
       colors=colors_pie,
       autopct='%1.1f%%',
       startangle=90,
       textprops={'fontsize': 11})
ax.set_title('Training Health Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Visualizations completed!")

## Step 9: Detailed Issue Analysis

In [ ]:
print("\n" + "="*80)
print("🔍 DETAILED ISSUE ANALYSIS")
print("="*80 + "\n")

all_issues = {}
all_recommendations = {}

for report in monitor.reports:
    for issue in report.issues:
        all_issues[issue] = all_issues.get(issue, 0) + 1
    for rec in report.recommendations:
        all_recommendations[rec] = all_recommendations.get(rec, 0) + 1

if all_issues:
    print("📋 Issues Detected:")
    for issue, count in sorted(all_issues.items(), key=lambda x: x[1], reverse=True):
        print(f"   • {issue}: {count} times")
else:
    print("✅ No issues detected - Training is healthy!")

if all_recommendations:
    print("\n💡 Top Recommendations:")
    for i, (rec, count) in enumerate(sorted(all_recommendations.items(), key=lambda x: x[1], reverse=True)[:5], 1):
        print(f"   {i}. {rec}")

print("\n" + "="*80)

## Step 10: Generate Report

In [ ]:
# Generate comprehensive report
report = monitor.generate_report()

print("\n" + "="*80)
print("📄 COMPREHENSIVE REPORT")
print("="*80 + "\n")

import json
print(json.dumps({
    'summary': report['summary'],
    'total_reports': len(monitor.reports)
}, indent=2))

print("\n✅ Report generated successfully!")

## Summary & Next Steps

### What We Accomplished:
✅ Cloned and installed Training Health Monitor  
✅ Created a neural network model  
✅ Initialized the monitoring framework  
✅ Trained with real-time health monitoring  
✅ Detected issues and got recommendations  
✅ Generated visualizations and reports  

### Key Features Demonstrated:
- 🎯 **Real-time Detection**: Issues caught during training
- 💡 **Smart Recommendations**: Actionable solutions provided
- 📊 **Comprehensive Metrics**: Loss, accuracy, health tracking
- 📈 **Visualizations**: Easy-to-understand charts and graphs
- 🔍 **Detailed Analysis**: Issue frequency and recommendations

### Next Steps:
1. Try with **different hyperparameters**
2. Test with **TensorFlow** (modify code accordingly)
3. Experiment with **different thresholds**
4. Use on your **own datasets and models**

### Resources:
- 📚 [Full Documentation](https://github.com/mohmed1983mohme-cyber/training-health-monitor)
- 💻 [GitHub Repository](https://github.com/mohmed1983mohme-cyber/training-health-monitor)
- 🎓 [API Reference](https://github.com/mohmed1983mohme-cyber/training-health-monitor/docs)
